# Step 4: Object Storage with MinIO

Thus far, the "lake" has consisted of a folder on local disk. In production, data lakes typically reside on **object storage** (S3, Azure Blob, GCS, etc.): storage and compute operate as separate services, scale independently, and can be shared by multiple engines simultaneously.

[MinIO](https://min.io) is an S3-compatible object store that can be run as a container directly within the Codespace, exposing the same API as AWS S3 and requiring no cloud account.

**Before running this notebook**, start MinIO from a terminal:

```bash
bash scripts/start_minio.sh
```

Codespaces forwards port **9001** automatically. Open the "Ports" tab and select the link to access the MinIO web console (login: `minioadmin` / `minioadmin`).

In [ ]:
import os
from pathlib import Path

while not (Path.cwd() / "requirements.txt").exists():
    os.chdir("..")
print("Working directory:", Path.cwd())

## Creating a Bucket and Uploading the Parquet Files

A bucket is the S3 equivalent of a top-level folder. The partitioned Parquet files from `lake/verkauf/` are uploaded to a bucket named `datalake`, preserving the `jahr=.../monat=.../data.parquet` key structure — partitioning functions identically on object storage.

In [ ]:
from minio import Minio

MINIO_ENDPOINT = "localhost:9000"
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin"
BUCKET = "datalake"

client = Minio(
    MINIO_ENDPOINT,
    access_key=MINIO_ACCESS_KEY,
    secret_key=MINIO_SECRET_KEY,
    secure=False,
)

if not client.bucket_exists(BUCKET):
    client.make_bucket(BUCKET)
    print(f"Created bucket '{BUCKET}'")
else:
    print(f"Bucket '{BUCKET}' already exists")

In [ ]:
local_root = Path("lake/verkauf")
uploaded = 0

for file in local_root.rglob("*.parquet"):
    object_key = "verkauf/" + file.relative_to(local_root).as_posix()
    client.fput_object(BUCKET, object_key, str(file))
    uploaded += 1

print(f"Uploaded {uploaded} files to s3://{BUCKET}/verkauf/")

Opening the MinIO console (port 9001) at this point shows the `datalake` bucket with the same `jahr=.../monat=.../` folder structure, visible directly in the browser.

## Querying MinIO Directly from DuckDB

DuckDB's `httpfs` extension implements the S3 API. `CREATE SECRET` registers the MinIO credentials and endpoint once per session; thereafter, `s3://...` paths function exactly like local paths.

In [ ]:
import duckdb

con = duckdb.connect()
con.sql("INSTALL httpfs")
con.sql("LOAD httpfs")

con.sql(f"""
    CREATE SECRET minio_secret (
        TYPE S3,
        KEY_ID '{MINIO_ACCESS_KEY}',
        SECRET '{MINIO_SECRET_KEY}',
        ENDPOINT '{MINIO_ENDPOINT}',
        URL_STYLE 'path',
        USE_SSL false,
        REGION 'us-east-1'
    )
""")
print("Secret registered.")

In [ ]:
con.sql("""
    SELECT region, SUM(revenue) AS total_revenue
    FROM 's3://datalake/verkauf/**/*.parquet'
    GROUP BY region
    ORDER BY total_revenue DESC
""").show()

## Key Takeaway

Comparing this query to the one in `03_duckdb_queries.ipynb` shows that it is identical except for the path — `lake/verkauf/...` has become `s3://datalake/verkauf/...`. DuckDB, the compute/query engine, remains unchanged; only the underlying storage layer differs.

Concretely, storage and compute are two separate processes here, communicating only over the network:

- **Storage is MinIO** — a separate process (a container, started beforehand via `bash scripts/start_minio.sh`), listening on `localhost:9000` for the S3 API. It holds the actual Parquet bytes inside the `datalake` bucket and does nothing but serve them on request.
- **Compute is DuckDB** — running inside this notebook's own Python kernel, a different process from MinIO entirely. It parses the SQL, plans the query, applies predicate pushdown and column pruning, and performs the aggregation.
- **The link between them** is the `httpfs` extension, which speaks the S3 protocol over HTTP to the endpoint registered in `CREATE SECRET`. DuckDB requests only the bytes it needs; MinIO has no awareness of SQL, query plans, or aggregation.

This is what makes the separation real rather than conceptual: killing the notebook's kernel would leave MinIO and the data untouched, and stopping MinIO would leave DuckDB fully intact as a query engine, just with nothing left to query. The same engine can query a local folder, an S3 bucket, or, as in production, a cloud data lake, with no change required to the SQL.